# Пайплайн

Основная идея пайплайна:

1. грузим датасеты
2. удаляем несколько самых больших выбросов по SI
3. чистим признаки и убираем константные / сильно коррелирующие колонки
4. обучаем базовый ансамбль CatBoost + LightGBM для IC50, CC50, SI
5. сохраняем out-of-fold и test прогнозы в кэш
6. отдельно делаем стекинг для SI через несколько LightGBM-моделей
7. для предсказаний этих моделей делаем постобработку, если модели не сходятся в предсказаниях
8. собираем все результаты воедино в submission.csv.


In [1]:
import os
import re
import time

import lightgbm as lgb
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold, StratifiedKFold

# Конфиг

Довольно долго проверять модели, если собирать параметры на местах, поэтому мы вынесли их в отдельный блок.

In [2]:
PATH_BASE_CACHE = 'multiseed_base.npz'

TARGETS = ['IC50', 'CC50', 'SI']

N_TOP_SI_DROP = 5  # сколько хотим самых экстремальных SI игнорировать при обучении

CORR_CUT = 0.95  # предел порога корреляции для признаков

# Настройки базового ансамбля

N_FOLDS_BASE = 10
MODEL_WEIGHTS = {
    'catboost': 0.75,
    'lightgbm': 0.25,
}
SEEDS_BASE = (42, 1337, 2024)

# Настройки стекинга для SI
N_FOLDS_STACK = 5
SEEDS_STACK = (42, 1337, 2024)

# Настройки постобработки
ADAPTIVE_TOP_PCT = 3.5
CLIP_PCT_IC50 = 98.80
CLIP_PCT_CC50 = 98.95

CATBOOST_PARAMS = {
    'IC50': dict(iterations=2000, learning_rate=0.015, depth=7, l2_leaf_reg=7.0, subsample=0.8),
    'CC50': dict(iterations=2000, learning_rate=0.015, depth=9, l2_leaf_reg=9.0, subsample=0.7),
    'SI': dict(iterations=2000, learning_rate=0.015, depth=7, l2_leaf_reg=6.0, subsample=0.9),
}
LIGHTGBM_PARAMS = {
    'IC50': dict(n_estimators=2000, learning_rate=0.015, max_depth=7, subsample=0.8, colsample_bytree=0.8),
    'CC50': dict(n_estimators=2000, learning_rate=0.015, max_depth=9, subsample=0.7, colsample_bytree=0.7),
    'SI': dict(n_estimators=2000, learning_rate=0.03, max_depth=7, subsample=0.9, colsample_bytree=0.9),
}

V2C_VARIANTS = {
    'v0': dict(objective='rmse', learning_rate=0.020, num_leaves=15, n_estimators=4000,
               min_data_in_leaf=20, subsample=0.85, subsample_freq=1, colsample_bytree=0.5,
               reg_lambda=1.0, verbose=-1),
    'v1': dict(objective='rmse', learning_rate=0.015, num_leaves=10, n_estimators=4000,
               min_data_in_leaf=25, subsample=0.80, subsample_freq=1, colsample_bytree=0.4,
               reg_lambda=2.0, verbose=-1),
    'v2': dict(objective='rmse', learning_rate=0.025, num_leaves=20, n_estimators=4000,
               min_data_in_leaf=15, subsample=0.90, subsample_freq=1, colsample_bytree=0.6,
               reg_lambda=0.5, verbose=-1),
    'v3': dict(objective='huber', alpha=0.9, learning_rate=0.020, num_leaves=15, n_estimators=4000,
               min_data_in_leaf=20, subsample=0.85, subsample_freq=1, colsample_bytree=0.5,
               reg_lambda=1.0, verbose=-1),
}

# Загрузка данных

Сразу проверяем размерности и названия колонок, чтобы не ловить ошибки через час обучения, и переименуем колонки-таргеты, чтобы они совпадали с требованием к решению.

In [3]:
train_full = pd.read_csv('train.csv', index_col='index')
test = pd.read_csv('test.csv', index_col='index')
sample = pd.read_csv('sample_submission.csv', index_col='index')

train_full = train_full.rename(
    columns={target_orig: target for target_orig, target in zip(['IC50, mM', 'CC50, mM', 'SI'], TARGETS)})

print('train:', train_full.shape)
print('test:', test.shape)
print('sample:', sample.shape)

display(train_full.head())
display(test.head())
display(sample.head())

train: (751, 213)
test: (250, 210)
sample: (250, 3)


,IC50,CC50,SI,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
index,,,,,,,,,,,,,,,,,,,,,
0,102.414420,95.757483,0.935000,5.466584,5.466584,0.719259,0.719259,0.681165,18.307692,195.287,...,1,0,0,0,0,0,0,0,0,0
1,0.044333,8.401080,189.500000,11.492712,11.492712,0.012350,-3.798024,0.769122,27.652174,360.907,...,0,1,0,0,0,0,0,0,0,0
2,4.437964,50.085589,11.285714,5.366084,5.366084,0.522930,0.522930,0.612606,24.608696,315.457,...,0,0,0,0,0,0,0,0,0,0
3,6.827881,682.788051,100.000000,13.317130,13.317130,0.020658,-4.829339,0.345823,12.400000,439.375,...,0,0,1,0,0,0,0,0,0,0
4,2.003253,70.001455,34.943894,6.320833,6.320833,0.300347,0.300347,0.562066,60.272727,151.253,...,0,0,0,0,0,0,0,0,0,0


,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
index,,,,,,,,,,,,,,,,,,,,,
0,13.761882,13.761882,0.121946,-0.962625,0.770057,30.580645,450.541,432.397,450.070799,156,...,1,0,0,0,0,0,0,1,0,0
1,13.224489,13.224489,0.066132,-1.801871,0.278628,25.687500,448.380,428.220,448.100561,170,...,0,0,0,0,0,0,0,0,0,0
2,6.191528,6.191528,0.445278,0.445278,0.657472,55.384615,179.307,158.139,179.167400,74,...,0,0,0,0,0,0,0,0,0,0
3,14.061236,14.061236,0.054870,-6.660336,0.564307,23.464286,410.289,397.185,410.086525,152,...,0,0,0,0,0,0,0,0,0,0
4,12.790378,12.790378,0.320463,-1.642616,0.696213,22.000000,280.279,268.183,280.073559,104,...,0,0,0,0,0,0,0,0,0,0


,IC50,CC50,SI
index,,,
0,0.723678,0.426328,0.308486
1,0.804617,0.643166,0.179750
2,0.907425,0.518521,0.298384
3,0.563266,0.054770,0.280019
4,0.935028,0.616856,0.979752


# Подготовка признаков


LightGBM часто ругается на названия колонок, поэтому мы их чистим от спецсимволов.

In [4]:
def sanitize_columns(df):
    column_names_map = {
        c: re.sub(r'[^A-Za-z0-9_]+', '__', c)
        for c in df.columns
    }
    return df.rename(columns=column_names_map)

Убираем следующие фичи:
 - константы
 - фичи с коэффициентом корреляции выше 0.95, чтобы одни и те же фичи не портили модели

Матрица корреляции на 200+ признаков выглядит очень вырвиглазно даже при figsize=(64, 48), поэтому выводить не будем, просто проверим по условию.

In [5]:
def remove_constant_features(X_train, X_test):
    column_variance = X_train.var()
    columns_to_keep = column_variance[column_variance > 1e-8].index
    X_train = X_train[columns_to_keep]
    X_test = X_test[columns_to_keep]
    return X_train, X_test


def remove_correlated_features(X_train, X_test, corr_cut=CORR_CUT):
    corr = X_train.corr().abs()
    upper_triangle_mask = np.triu(
        np.ones(corr.shape, dtype=bool),
        k=1
    )

    upper_triangle = corr.where(upper_triangle_mask)
    columns_to_drop = [
        c
        for c in upper_triangle.columns
        if (upper_triangle[c] > corr_cut).any()
    ]

    X_train = X_train.drop(columns=columns_to_drop)
    X_test = X_test.drop(columns=columns_to_drop)

    return X_train, X_test


def select_features(X_train, X_test, corr_cut=CORR_CUT):
    X_train, X_test = remove_constant_features(X_train, X_test)
    X_train, X_test = remove_correlated_features(X_train, X_test, corr_cut)
    return X_train, X_test

Убираем самые большие SI, так как из-за использования RMSE они очень сильно искажают модель.

In [6]:
def remove_extreme_si(train_df):
    extreme_si_idx = train_df.nlargest(N_TOP_SI_DROP, "SI").index
    keep_row_idx = ~train_df.index.isin(extreme_si_idx)
    clean_train_idx = train_df.index[keep_row_idx].values
    clean_train = train_df.loc[keep_row_idx].reset_index(drop=True)

    return clean_train, clean_train_idx

Общая функция-пайплайн, прогоняющая все подготовительные шаги

In [7]:
def prepare_features(train_full, test):
    clean_train, clean_train_idx = remove_extreme_si(train_full)
    X_train = sanitize_columns(clean_train.drop(columns=TARGETS))
    X_test = sanitize_columns(test)
    X_train, X_test = select_features(X_train, X_test)
    return clean_train, clean_train_idx, X_train, X_test

Проверим, сколько признаков остаётся после подготовки.


In [8]:
clean_train, clean_train_idx, train_feature_df, X_test = prepare_features(train_full, test)

print('До удаления выбросов:', len(train_full))
print('После удаления выбросов:', len(clean_train))
print('Признаков после отбора:', train_feature_df.shape[1])

train_feature_df.head()

До удаления выбросов: 751
После удаления выбросов: 746
Признаков после отбора: 159


,MaxAbsEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,MaxPartialCharge,MinPartialCharge,MaxAbsPartialCharge,FpDensityMorgan1,...,fr_quatN,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiophene,fr_unbrch_alkane,fr_urea
0,5.466584,0.719259,0.719259,0.681165,18.307692,195.287,0.119177,-0.360247,0.360247,1.230769,...,0,1,0,0,0,0,0,0,0,0
1,11.492712,0.012350,-3.798024,0.769122,27.652174,360.907,0.237676,-0.393087,0.393087,1.304348,...,0,0,1,0,0,0,0,0,0,0
2,5.366084,0.522930,0.522930,0.612606,24.608696,315.457,0.160487,-0.492870,0.492870,1.173913,...,0,0,0,0,0,0,0,0,0,0
3,13.317130,0.020658,-4.829339,0.345823,12.400000,439.375,0.436923,-0.274885,0.436923,1.033333,...,0,0,0,1,0,0,0,0,0,0
4,6.320833,0.300347,0.300347,0.562066,60.272727,151.253,0.016184,-0.325030,0.325030,0.818182,...,0,0,0,0,0,0,0,0,0,0


# Ансамбль CatBoost + LightGBM

Для каждого таргета обучается отдельный ансамбль: бленд catboost и lightgbm (75/25). Для обучения используются 10 фолдов на 3 разных рандомных сидах, чтобы общие предсказания были более стабильными.

Важно - SI логарифмируется, иначе масштаб улетает в небеса, что очень сильно перекашивает модель.

In [9]:
BASE_CACHE_DIR = "base_cache"

PATH_CLEAN_INDICES = f"{BASE_CACHE_DIR}/clean_indices.csv"
PATH_OOF_PREDICTIONS = f"{BASE_CACHE_DIR}/oof_predictions.csv"
PATH_TEST_PREDICTIONS = f"{BASE_CACHE_DIR}/test_predictions.csv"


def base_cache_exists():
    return (
            os.path.exists(PATH_CLEAN_INDICES)
            and os.path.exists(PATH_OOF_PREDICTIONS)
            and os.path.exists(PATH_TEST_PREDICTIONS)
    )


def save_base_cache(base):
    os.makedirs(BASE_CACHE_DIR, exist_ok=True)

    pd.Series(
        base["clean_indices"],
        name="train_index",
    ).to_csv(PATH_CLEAN_INDICES, index=False)

    pd.DataFrame({
        target: base[f"oof_{target}"]
        for target in TARGETS
    }).to_csv(PATH_OOF_PREDICTIONS, index=False)

    pd.DataFrame({
        target: base[f"pred_{target}"]
        for target in TARGETS
    }).to_csv(PATH_TEST_PREDICTIONS, index=False)


def load_base_cache():
    clean_indices = pd.read_csv(PATH_CLEAN_INDICES)["train_index"].to_numpy()

    oof_predictions = pd.read_csv(PATH_OOF_PREDICTIONS)
    test_predictions = pd.read_csv(PATH_TEST_PREDICTIONS)

    base = {
        "clean_indices": clean_indices,
    }

    for target in TARGETS:
        base[f"oof_{target}"] = oof_predictions[target].to_numpy()
        base[f"pred_{target}"] = test_predictions[target].to_numpy()

    return base

In [10]:
def train_multiseed_base(train_full, test):
    if base_cache_exists():
        return load_base_cache()

    clean_train, clean_train_idx, train_features, X_test = prepare_features(train_full, test)

    oof_acc = {t: [] for t in TARGETS}
    pred_acc = {t: [] for t in TARGETS}
    start_time = time.time()
    for seed_idx, seed in enumerate(SEEDS_BASE, start=1):
        print(f'starting seed {seed_idx}/{len(SEEDS_BASE)}... {time.time() - start_time:.2f}s elapsed')
        kf = KFold(n_splits=N_FOLDS_BASE, shuffle=True, random_state=seed)
        oof_seed = {t: np.zeros(len(clean_train)) for t in TARGETS}
        pred_seed = {t: np.zeros(len(test)) for t in TARGETS}

        for target_idx, target in enumerate(TARGETS, start=1):
            print(
                f'  starting target {target_idx}/{len(TARGETS)}: {target} ... {time.time() - start_time:.2f}s elapsed')
            y = clean_train[target].astype(float).values
            use_log = (target == 'SI')
            y_fit = np.log1p(y) if use_log else y

            for fold_idx, (train_idx, validation_idx) in enumerate(kf.split(train_features), start=1):
                # print(f'    starting fold {fold_idx}/{N_FOLDS_BASE}... {time.time() - start_time:.2f}s elapsed')
                X_train, X_validation = train_features.iloc[train_idx], train_features.iloc[validation_idx]
                y_train, y_validation = y_fit[train_idx], y_fit[validation_idx]

                # catboost
                catboost = CatBoostRegressor(
                    **CATBOOST_PARAMS[target], eval_metric='RMSE',
                    random_seed=seed, verbose=0, allow_writing_files=False
                )
                catboost.fit(X_train, y_train, eval_set=(X_validation, y_validation), early_stopping_rounds=100)
                validation_pred_catboost = catboost.predict(X_validation)
                test_pred_catboost = catboost.predict(X_test)

                # lightbgm
                lightbgm = lgb.LGBMRegressor(**LIGHTGBM_PARAMS[target], random_state=seed,
                                             n_jobs=-1, verbose=-1)
                lightbgm.fit(X_train, y_train, eval_set=[(X_validation, y_validation)],
                             callbacks=[lgb.early_stopping(100, verbose=False)])
                validation_pred_lightbgm = lightbgm.predict(X_validation)
                test_pred_lightbgm = lightbgm.predict(X_test)

                # blend
                validation_pred_blend = MODEL_WEIGHTS['catboost'] * validation_pred_catboost \
                                        + MODEL_WEIGHTS['lightgbm'] * validation_pred_lightbgm
                test_pred_blend = MODEL_WEIGHTS['catboost'] * test_pred_catboost \
                                  + MODEL_WEIGHTS['lightgbm'] * test_pred_lightbgm

                if use_log:
                    validation_pred_blend, test_pred_blend = np.expm1(validation_pred_blend), np.expm1(test_pred_blend)

                validation_pred_blend = np.clip(validation_pred_blend, 0, None)
                test_pred_blend = np.clip(test_pred_blend, 0, None)

                # важно - каждый валидационный фолд предсказывается только теми моделями, которые его не видели
                oof_seed[target][validation_idx] = validation_pred_blend
                pred_seed[target] += test_pred_blend / N_FOLDS_BASE

        for t in TARGETS:
            oof_acc[t].append(oof_seed[t])
            pred_acc[t].append(pred_seed[t])

    base = {'clean_indices': clean_train_idx}
    for t in TARGETS:
        base[f'oof_{t}'] = np.mean(np.stack(oof_acc[t]), axis=0)
        base[f'pred_{t}'] = np.mean(np.stack(pred_acc[t]), axis=0)

    save_base_cache(base)
    print(f'finished in {time.time() - start_time:.2f}s')
    return base

# Запуск обучения базового ансамбля

In [11]:
base = train_multiseed_base(train_full, test)
base.keys()

starting seed 1/3... 0.00s elapsed
  starting target 1/3: IC50 ... 0.00s elapsed
  starting target 2/3: CC50 ... 52.19s elapsed
  starting target 3/3: SI ... 250.07s elapsed
starting seed 2/3... 287.70s elapsed
  starting target 1/3: IC50 ... 287.70s elapsed
  starting target 2/3: CC50 ... 332.91s elapsed
  starting target 3/3: SI ... 561.24s elapsed
starting seed 3/3... 600.83s elapsed
  starting target 1/3: IC50 ... 600.83s elapsed
  starting target 2/3: CC50 ... 644.95s elapsed
  starting target 3/3: SI ... 889.00s elapsed
finished in 923.21s


dict_keys(['clean_indices', 'oof_IC50', 'pred_IC50', 'oof_CC50', 'pred_CC50', 'oof_SI', 'pred_SI'])

# Сбор матрицы для стекинга SI

Стекинг используется только для SI, потому что даже огромного базового ансамбля не хватает для точных предсказаний.

Для этого к исходным фичам подмешиваются предсказания первого ансамбля, и они участвуют в обучении и предсказаниях второго набора моделей.

Функция обучения ансамбля для SI, принцип примерно тот же, что у первого ансамбля - валидационные предикты только на фолде, на котором модель не училась, и усреднённый предикт тестовой выборки.

In [12]:
def train_v2c_variants(X_train, X_test, si_log):
    strat_bins = pd.qcut(si_log, 10, labels=False, duplicates='drop')
    pred_by_variant = {name: np.zeros(X_test.shape[0]) for name in V2C_VARIANTS}

    for name, params in V2C_VARIANTS.items():
        for seed in SEEDS_STACK:
            kf = StratifiedKFold(n_splits=N_FOLDS_STACK, shuffle=True, random_state=seed)

            for train_idx, validation_idx in kf.split(X_train, strat_bins):
                model = lgb.LGBMRegressor(random_state=seed, **params)
                model.fit(
                    X_train.iloc[train_idx],
                    si_log.iloc[train_idx],
                    eval_set=[
                        (
                            X_train.iloc[validation_idx],
                            si_log.iloc[validation_idx]
                        )
                    ],
                    callbacks=[lgb.early_stopping(150, verbose=False)]
                )
                pred_by_variant[name] += np.expm1(model.predict(X_test)) / (len(SEEDS_STACK) * N_FOLDS_STACK)

        pred_by_variant[name] = np.maximum(pred_by_variant[name], 1e-3)

    teachers = np.stack([pred_by_variant[n] for n in V2C_VARIANTS])
    return teachers

In [13]:
clean_train = train_full.iloc[base['clean_indices']].reset_index(drop=True)

feature_cols = [c for c in clean_train.columns if c not in TARGETS]

train_features = clean_train[feature_cols].astype(float)
test_features = test[feature_cols].astype(float)

# заполняем пробелы медианами из train
train_medians = train_features.median()
train_features = train_features.fillna(train_medians)
test_features = test_features.fillna(train_medians)

# удаляем почти константные признаки
train_features, test_features = remove_constant_features(train_features, test_features)

X_stack_train = train_features.copy()
X_stack_train['pred_IC50'] = base['oof_IC50']
X_stack_train['pred_CC50'] = base['oof_CC50']
X_stack_train['pred_SI'] = base['oof_SI']

X_stack_test = test_features.copy()
X_stack_test['pred_IC50'] = base['pred_IC50']
X_stack_test['pred_CC50'] = base['pred_CC50']
X_stack_test['pred_SI'] = base['pred_SI']

si_log = pd.Series(np.log1p(clean_train['SI'].astype(float).clip(lower=0)))

print(X_stack_train.shape, X_stack_test.shape)

(746, 195) (250, 195)


# Обучение стекинга для SI

Обучаем 4 варианта LightGBM с разными параметрами. Каждый вариант обучается на 3 сидах и 5 стратфолдах по логарифму.

In [14]:
teacher_stack = train_v2c_variants(X_stack_train, X_stack_test, si_log)

In [15]:
# транспонируем, чтобы модели были по столбцам, а записи по строкам
# тогда размерность получается <кол-во записей в тесте> х <кол-во моделей>
stacking_model_pred = pd.DataFrame(teacher_stack.T, columns=list(V2C_VARIANTS.keys()))

# print('v2c_mean:', v2c_mean.shape)
print('teachers_te:', stacking_model_pred.shape)
stacking_model_pred.head()

teachers_te: (250, 4)


,v0,v1,v2,v3
0,4.587041,4.333816,4.619982,4.273311
1,3.997394,3.790913,3.941952,4.153709
2,6.321044,6.013992,6.766631,5.978031
3,4.357830,3.805105,4.615474,4.294328
4,2.451381,2.452386,2.282661,2.137472


# Постобработка

Постобработка делает две вещи:

1. для самых “неуверенных” объектов по SI (дисперсия предсказаний выше порога) берёт минимальный прогноз среди teacher-моделей;
2. клипает верхние хвосты IC50 и CC50 по перцентилям, у нас экспериментально подобраны 98.8% и 99.95% соответственно.


In [16]:
def adaptive_replace_uncertain(model_predictions, top_pct=ADAPTIVE_TOP_PCT):
    prediction_log = np.log1p(model_predictions.clip(lower=1e-3))

    disagreement = prediction_log.std(axis=1)
    disagreement_threshold = np.percentile(disagreement, 100.0 - top_pct)
    uncertain_idx = disagreement >= disagreement_threshold

    out = model_predictions.mean(axis=1)
    out[uncertain_idx] = model_predictions.min(axis=1)[uncertain_idx]
    return np.maximum(out, 1e-3)


def build_submission(pred_ic, pred_cc, pred_si, sample):
    ic50_clip_threshold = float(np.percentile(pred_ic, CLIP_PCT_IC50))
    cc50_clip_threshold = float(np.percentile(pred_cc, CLIP_PCT_CC50))
    ic50_prediction = np.clip(pred_ic, 0, ic50_clip_threshold)
    cc50_prediction = np.clip(pred_cc, 0, cc50_clip_threshold)
    return pd.DataFrame({
        'index': sample.index,
        'IC50': np.maximum(ic50_prediction, 1e-3),
        'CC50': np.maximum(cc50_prediction, 1e-3),
        'SI': np.maximum(pred_si, 1e-3),
    })

In [17]:
si_final = adaptive_replace_uncertain(stacking_model_pred)

# Сбор посылки на лидерборд

In [18]:
sub = build_submission(
    base['pred_IC50'],
    base['pred_CC50'],
    si_final,
    sample,
)

display(sub.head())
display(sub.describe())

os.makedirs('submissions', exist_ok=True)
sub.to_csv('submissions/submission.csv', index=False)

,index,IC50,CC50,SI
0,0,191.609595,411.053007,4.453537
1,1,211.107139,394.703490,3.970992
2,2,74.935683,283.134262,6.269925
3,3,295.121313,458.670300,4.268184
4,4,197.641801,337.415315,2.330975


,index,IC50,CC50,SI
count,250.000000,250.000000,250.000000,250.000000
mean,124.500000,235.221177,635.593289,14.236541
std,72.312977,249.173787,476.411856,29.025430
min,0.000000,48.539912,95.540162,1.346330
25%,62.250000,95.522729,331.261304,3.527925
50%,124.500000,154.813323,481.631228,5.462945
75%,186.750000,269.381126,831.411418,10.524186
max,249.000000,1593.912468,2596.672310,256.873704
